# SkillOutcome SIH26135 ML workflow

This notebook inspects normalized snapshots and reproducible model-comparison metrics. Run data preparation and training first.

In [2]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed' / 'placement_training_snapshot.csv'
REPORTS = ROOT / 'reports'
df = pd.read_csv(DATA)
df.shape, df['Trainee_ID'].nunique(), df[['Trainee_ID', 'Target_Job_Role', 'Skill_Gap_Score']].head()

FileNotFoundError: [Errno 2] No such file or directory: '/data/processed/placement_training_snapshot.csv'

In [ ]:
cohort_summary = df[['Placement_Target', 'Attendance_Percent', 'Assessment_Score', 'Training_Performance', 'Skill_Gap_Score', 'Demand_Score']].describe().T
cohort_summary

In [ ]:
for name in ['placement_model_metrics.json', 'attrition_model_metrics.json']:
    metrics = json.loads((REPORTS / name).read_text(encoding='utf-8'))
    print(name, metrics['selected_algorithm'], metrics['test_metrics'])

## Model comparison

Every candidate is scored on the chronological validation split; the selected algorithm is the winner on the task's selection metric.

In [ ]:
comparison = {}
for name in ['placement_model_metrics.json', 'attrition_model_metrics.json']:
    metrics = json.loads((REPORTS / name).read_text(encoding='utf-8'))
    comparison[metrics['model_name']] = pd.DataFrame(metrics['model_comparison']).T.assign(selected=lambda frame: frame.index == metrics['selected_algorithm'])
comparison['placement_model']

In [ ]:
comparison['attrition_model']

## Attrition risk bands

Termination is the rare class, so accuracy at 0.5 is uninformative. Read average precision and the metrics at the production thresholds.

In [ ]:
attrition_metrics = json.loads((REPORTS / 'attrition_model_metrics.json').read_text(encoding='utf-8'))
print('label:', attrition_metrics['target_description'])
print('thresholds:', attrition_metrics['risk_thresholds'])
print('calibration:', attrition_metrics['calibration'])
pd.DataFrame({
    'at_0.5': attrition_metrics['test_metrics'],
    'at_medium': attrition_metrics.get('test_metrics_at_medium_risk_threshold', {}),
    'at_high': attrition_metrics.get('test_metrics_at_high_risk_threshold', {}),
}).drop(index=['confusion_matrix'], errors='ignore')

In [ ]:
def band_table(summary):
    """reports/*_metrics.json stores bands as {'positive_label': ..., 'bands': {band: {...}}}."""
    if not summary:
        return pd.DataFrame()
    return pd.DataFrame(summary['bands']).T.rename_axis('band').reset_index()

In [ ]:
band_table(attrition_metrics.get('risk_band_distribution_test'))

## Placement support priority

Roughly 86% of completers are placed, so a 0.5 cut-off flags almost nobody who needs help. The support thresholds are fitted on the non-placement side of the calibration split, and priority rises as placement probability falls.

In [ ]:
placement_metrics = json.loads((REPORTS / 'placement_model_metrics.json').read_text(encoding='utf-8'))
print('support thresholds:', placement_metrics['support_thresholds'])
print('calibration:', placement_metrics['calibration'])
pd.DataFrame({
    'at_0.5_placement': placement_metrics['test_metrics'],
    'non_placement_at_medium': placement_metrics.get('test_metrics_at_medium_support_threshold', {}),
    'non_placement_at_high': placement_metrics.get('test_metrics_at_high_support_threshold', {}),
}).drop(index=['confusion_matrix'], errors='ignore')

In [ ]:
band_table(placement_metrics.get('support_band_distribution_test'))

## Data quality and leakage checks

The preparation run records the label definition, the observation windows, and the checks it enforced before committing the tables.

In [ ]:
quality = json.loads((REPORTS / 'data_quality.json').read_text(encoding='utf-8'))
print(json.dumps({k: quality[k] for k in ['rows', 'labels', 'snapshot_windows', 'feature_completeness', 'checks_passed']}, indent=2))

## Interpretation guardrail

Placement inputs exclude post-outcome columns. The normalized timeline is deterministic synthetic derivation because the supplied example has no actual employment-event history; replace it with operational extracts before production use.